# 02. Controlling Model Behaviour (Instructions & Temperature)

This notebook (**AI-103 -> Section 01**) builds on the first API call by introducing `instructions` (a system-prompt equivalent) and `temperature` (a creativity/randomness knob), and switches auth from a raw API key to **Microsoft Entra ID** -- the pattern every remaining script in this section uses.

**Difficulty: Beginner/Intermediate**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `openai`
- `azure-identity` -- provides `DefaultAzureCredential` and `get_bearer_token_provider` for Entra ID (Azure AD) authentication.
- `python-dotenv`

**Azure resources**
- The same Azure OpenAI / AI Foundry deployment as notebook 01 (e.g. `gpt-4.1`).
- You must be signed in with `az login` (or another credential the `DefaultAzureCredential` chain can discover), and that identity needs an RBAC role on the resource such as **Cognitive Services OpenAI User** -- no API key is used in this notebook.

**Environment variables** -- add to the root `.env`:
```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_DEPLOYMENT=gpt-4.1
```


## What You'll Learn
- The Entra ID / `DefaultAzureCredential` + `get_bearer_token_provider` auth pattern
- `instructions` -- the Responses API's equivalent of a system message
- `temperature` -- controlling output randomness/creativity
- `max_output_tokens` -- capping generation length (shown commented out, as in the original)
- Why this auth pattern is the one recommended for production Azure OpenAI apps


### 1. Imports, configuration, and the Entra ID token provider

`DefaultAzureCredential()` tries a chain of credential sources (environment variables, managed identity, the Azure CLI session, VS Code, etc.) and uses whichever succeeds first -- in local development this is typically your `az login` session. `get_bearer_token_provider(...)` wraps that credential into a zero-argument callable that the SDK invokes fresh on every request, so you never have to manually track token expiry/refresh yourself.

**Exam tip:** The scope `https://ai.azure.com/.default` requests a token valid for Azure AI Foundry / Cognitive Services resources. AI-102 tests whether you know that Entra ID tokens are short-lived and scoped, unlike a static API key -- this is precisely why Microsoft recommends it for production workloads (smaller blast radius if leaked, centrally revocable via RBAC).

**Alternatives:** API-key auth, as shown in notebook 01 -- simpler to set up for a quick demo, but the key itself is a long-lived secret you must store and rotate yourself.


In [ ]:
import os

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")


### 2. Build the client

Notice `api_key=token_provider` -- not a string. The `openai` SDK's `api_key` parameter accepts either a plain string **or** a zero-argument callable, and calls it fresh for every request. That's what makes the Entra ID pattern work directly with the base `OpenAI` client against Azure's v1 API surface, instead of requiring the dedicated `AzureOpenAI` client's `azure_ad_token_provider` field.


In [ ]:
client = OpenAI(
    base_url=endpoint,
    api_key=token_provider,
)


### 3. Send a request with `instructions` and `temperature`

`instructions` sets the assistant's persona/behaviour for this call ("You are a creative copywriter.") -- functionally similar to a `system` role message in Chat Completions, but applied per-call rather than stored as part of message history. `temperature=2` is the **maximum** allowed value here, deliberately chosen to show highly varied, creative output; `max_output_tokens` is left commented out, exactly as in the original script, so you can uncomment it yourself in the exercises below.

**Exam tip:** `temperature` typically ranges 0-2: low values (near 0) produce focused, deterministic output good for factual/structured tasks; high values (near 2) produce more random, creative, and less coherent output. For tasks needing consistent, parseable output (e.g. structured data extraction), prefer a low temperature -- or better, use structured outputs / a JSON schema constraint rather than relying on temperature alone. `max_output_tokens` caps the response length for cost and latency control; if generation is cut off by the cap, the response signals it was truncated rather than silently returning a complete answer.

**Alternatives:** `top_p` (nucleus sampling) is another randomness knob; OpenAI's own guidance is to adjust `temperature` **or** `top_p`, not both at once, since they interact.


In [ ]:
response = client.responses.create(
    model=deployment_name,
    instructions="You are a creative copywriter.",
    input="Write a two-sentence tagline for a new AI-powered productivity app.",
    # max_output_tokens=50,
    temperature=2,
)


### 4. Read the answer


In [ ]:
print(f"answer: {response.output_text}")


## Summary

You authenticated with Entra ID instead of an API key, then used `instructions` to set a persona and `temperature=2` to push the model toward maximally creative output. The printed tagline should read as noticeably more free-form/varied than a low-temperature call would produce.


## Try It Yourself
1. **Easy:** Lower `temperature` to `0.2` and re-run a few times -- compare how consistent the taglines are versus `temperature=2`.
2. **Medium:** Uncomment `max_output_tokens=50` and try a prompt that would naturally produce a longer answer; inspect the response to see how truncation is signaled.
3. **Advanced:** Loop over several `(instructions, temperature)` combinations, collect the outputs, and print them side by side to build a quick intuition for how each parameter shapes the result.
